### 1.1.14. Indexed Collections of Sets

An **indexed collection of sets** (also called an *indexed family of sets*) is a set-valued
entity that is indexed over another set. Where a scalar set has a single membership, an
indexed collection has one set per index value — each member of the indexing set picks
out a distinct component set.

The canonical example is a network: `set ARCS {NODES}` declares one arc-set per node,
so `ARCS[i]` is the set of nodes reachable from node `i`. Another common pattern is a
multi-product transportation model: `set dest {PROD}` declares one destination-set per
product, so `dest[p]` lists the destinations that product `p` can be shipped to.

Indexed set families are the AMPL mechanism for representing **ragged (irregular)
structures** — situations where the "inner" set differs in size or composition across
different values of the "outer" index. They are the natural alternative to a flat set of
tuples when the model has a hierarchical relationship between two levels of indexing.

In [ ]:
# Full grammar for an indexed set declaration (AMPL Reference Manual, A.6)
#
# set name { indexing } [ dimen n ] [ within sexpr ] [ = sexpr | default sexpr ] ;
#
# name      -- identifier for the collection; subscripted as name[i], name[i,j], etc.
# indexing  -- one or more index sets; defines the domain of the collection
#              e.g. {PROD}, {ORIG, DEST}, {p in PROD: condition}
# dimen n   -- (optional) fixes the dimension of every component set to n
#              if omitted and no 'within' is present, defaults to dimen 1
# within    -- (optional) constraint: every component set must be a subset of sexpr
#              sexpr may itself reference the indexing dummy variable(s)
# =         -- (optional) symbolic definition; component sets computed from sexpr;
#              data cannot override a set defined with =
# default   -- (optional) fallback value when no data is supplied for a component;
#              may be {} (the empty set); mutually exclusive with =
#
# Minimal forms:
set AREA {PROD};                          # one set per product; members from data
set ARCS {NODES};                         # one set per node; members from data

# With within:
set orig {PROD} within ORIG;              # orig[p] must be a subset of ORIG
set dest {PROD} within DEST;              # dest[p] must be a subset of DEST

# With =  (symbolic definition from expression):
set links {p in PROD} = orig[p] cross dest[p];   # links[p] computed, not from data

# With default:
set AREA2 {PROD} default {};              # empty set when data omits a component

# Multi-dimensional indexing:
set S {A, B};                             # one set per (a,b) pair; S[a,b]

# Component sets that are themselves compound (dimen 2):
set PAIR_ARCS {NODES} dimen 2 within NODES cross NODES;

#### Conceptual explanation

Let $I$ be a finite set (the index set) and let $U$ be a universe. An **indexed family of
sets** is a function $F : I \to 2^U$, where $2^U$ denotes the power set of $U$. For each
$i \in I$, the value $F[i]$ is a finite subset of $U$.

In AMPL, the declaration

```python
set ARCS {NODES};
```

introduces a function `ARCS` from `NODES` to sets of strings (or numbers). The component
`ARCS[i]` is the set associated with node `i`. Nothing requires `ARCS[i]` and `ARCS[j]`
to have the same members, the same cardinality, or any overlap — the structure is fully
**ragged**.

**When to use indexed families vs. flat tuple sets.**
Both representations encode the same mathematical information, but the choice of
representation affects how the model reads and how naturally the constraints are expressed.

| Criterion | Indexed family `set F {I}` | Flat tuple set `set T within I cross U` |
|---|---|---|
| Hierarchical relationship (I is "parent") | Natural | Awkward |
| Symmetric / peer relationship (I and U equivalent) | Awkward | Natural |
| Need `F[i]` repeatedly in the model | One subscript | Slice syntax each time |
| Need all `(i,j)` pairs at once | Use `setof` | Direct reference |
| Data varies per index value | Natural | Natural |

The AMPL book (Fourer, Gay, Kernighan, Chapter 6) observes: *"indexed collections are
most suitable for entities such as products and areas that have a hierarchical relationship.
Sets of tuples are preferable in dealing with entities like origins and destinations that are
related symmetrically."*

**Accessing a component.** The component set for index value `i` is written `ARCS[i]`.
If the index is a pair, write `S[a,b]`. The subscript syntax mirrors parameter subscripting
and follows the same rules: the number and types of subscripts must match the indexing
declaration.

**Cardinality.** `card(ARCS[i])` returns the number of elements in the component set
for node `i` as a scalar integer. This is valid anywhere a numeric expression is valid.

#### Attributes reference

The table below documents every attribute that may appear in an indexed set declaration.
All are optional except the indexing itself.

| Attribute | Syntax | Semantics |
|---|---|---|
| **Indexing** | `{I}`, `{I, J}`, `{i in I: cond}` | Defines the domain of the family; one component set per tuple in the index set. Required for an indexed collection. |
| **dimen** | `dimen n` | Every component set must have dimension `n` (i.e., contain `n`-tuples). Default: 1 when no `within` is present; inferred from the `within` expression otherwise. |
| **within** | `within sexpr` | Each component set must be a subset of `sexpr`. `sexpr` may reference the index dummy variable(s). Checked at data-load time; causes an informative error if violated. Multiple `within` phrases are allowed. |
| **=** | `= sexpr` | Symbolically defines each component; the expression is evaluated at model-generation time. Data cannot override a component defined with `=`. Mutually exclusive with `default`. |
| **default** | `default sexpr` | Specifies a fallback value for a component when no data is provided for that index value. May be `{}` (the empty set). Overridable by data or `let`. Mutually exclusive with `=`. |

#### Example 1 — Network model: outgoing arcs per node

The simplest and most important pattern. For each node in a directed graph, `ARCS[i]`
holds the set of nodes that can be reached directly from node `i`. Parameters and
variables indexed over `{i in NODES, j in ARCS[i]}` automatically cover only
the arcs that exist in the graph — no dummy "does this arc exist?" parameter is needed.

In [ ]:
# --- Model fragment: shortest-path / max-flow skeleton ---

set NODES;                              # all nodes in the directed graph
set ARCS {NODES} within NODES;         # ARCS[i] = nodes reachable from i
                                        # 'within NODES' validates each arc endpoint

param cap  {i in NODES, j in ARCS[i]} >= 0;   # arc capacity
param cost {i in NODES, j in ARCS[i]} >= 0;   # arc unit cost

var Flow {i in NODES, j in ARCS[i]} >= 0, <= cap[i,j];  # flow on arc (i,j)

# Cardinality: out-degree of each node
param outdeg {i in NODES} = card(ARCS[i]);

# Flow conservation at every node except source s and sink t:
param s symbolic in NODES;             # source node
param t symbolic in NODES;             # sink node

subject to Conservation {i in NODES: i <> s and i <> t}:
    sum {j in ARCS[i]} Flow[i,j]       # total outflow from i
  - sum {k in NODES: i in ARCS[k]} Flow[k,i]  # total inflow to i
  = 0;

**Data file for the network model.** Each component of the indexed family is supplied by
a separate `set` statement in the `.dat` file, with the index value as a subscript on the
set name.

In [ ]:
# --- Corresponding .dat file ---
data;

set NODES := a b c d e ;

# Each ARCS[i] is supplied individually:
set ARCS[a] := b c ;
set ARCS[b] := c d ;
set ARCS[c] := d e ;
set ARCS[d] := e ;
set ARCS[e] :=   ;   # empty: node e has no outgoing arcs

param s := a ;
param t := e ;

param cap :=
  [a,*]: b 10  c 8
  [b,*]: c 5   d 7
  [c,*]: d 6   e 9
  [d,*]: e 4
  ;

param cost :=
  [a,*]: b 1  c 2
  [b,*]: c 1  d 3
  [c,*]: d 2  e 1
  [d,*]: e 2
  ;

#### Example 2 — Multi-product transportation: per-product origin and destination sets

This is the `multic.mod` pattern from Fourer, Gay, Kernighan (Figure 6-5). Each product
may be supplied only from certain origins and demanded only at certain destinations.
Rather than a single sparse set of `(origin, destination, product)` triples, the model
uses three indexed families: `orig`, `dest`, and `links`. The `links` family is
**symbolically defined** — each `links[p]` is computed from `orig[p]` and `dest[p]`
via `cross`, so it never needs to be supplied in the data.

In [ ]:
# multic.mod  (Fourer, Gay, Kernighan, Figure 6-5)

set ORIG;                               # all possible origins
set DEST;                               # all possible destinations
set PROD;                               # products

set orig {PROD} within ORIG;            # orig[p]: origins supplying product p
set dest {PROD} within DEST;            # dest[p]: destinations demanding product p
set links {p in PROD} = orig[p] cross dest[p];  # links[p]: all (i,j) for product p

param supply {p in PROD, orig[p]} >= 0;         # supply at each origin for p
param demand {p in PROD, dest[p]} >= 0;         # demand at each destination for p

check {p in PROD}:
    sum {i in orig[p]} supply[p,i] = sum {j in dest[p]} demand[p,j];

param limit {ORIG, DEST} >= 0;                  # capacity of each (origin,dest) link
param cost  {p in PROD, links[p]} >= 0;         # shipping cost per unit

var Trans {p in PROD, links[p]} >= 0;           # units shipped

minimize Total_Cost:
    sum {p in PROD, (i,j) in links[p]} cost[p,i,j] * Trans[p,i,j];

subject to Supply {p in PROD, i in orig[p]}:
    sum {j in dest[p]} Trans[p,i,j] = supply[p,i];

subject to Demand {p in PROD, j in dest[p]}:
    sum {i in orig[p]} Trans[p,i,j] = demand[p,j];

# This constraint crosses the hierarchy: sum over products for a fixed (i,j) link.
subject to Multi {i in ORIG, j in DEST}:
    sum {p in PROD: (i,j) in links[p]} Trans[p,i,j] <= limit[i,j];

**Data file for multic.mod.** The `orig` and `dest` families are supplied with subscripted
`set` statements. The `links` family is defined with `=` in the model and therefore requires
no data statement.

In [ ]:
# --- multic.dat (abbreviated) ---
data;

set ORIG := GARY CLEV PITT ;
set DEST := FRA  DET  LAN  WIN  STL  FRE  LAF ;
set PROD := bands coils plate ;

# Per-product origin sets (ragged: each product has different origins)
set orig[bands] := GARY CLEV PITT ;
set orig[coils] := GARY CLEV ;
set orig[plate] := CLEV PITT ;

# Per-product destination sets
set dest[bands] := FRA  DET  LAN  WIN  STL  FRE  LAF ;
set dest[coils] := FRA  DET  LAN  WIN  STL  FRE  LAF ;
set dest[plate] := FRA  DET  LAN  WIN  STL  FRE  LAF ;

# 'links' is defined by = in the model; no data statement is needed or allowed.

param supply :=
  [bands,*]: GARY 400  CLEV 700  PITT 800
  [coils,*]: GARY 800  CLEV 700
  [plate,*]: CLEV 100  PITT 900
  ;

param demand :=
  [bands,*]: FRA 300  DET 300  LAN 100  WIN 75   STL 650  FRE 225  LAF 250
  [coils,*]: FRA 500  DET 750  LAN 400  WIN 250  STL 950  FRE 850  LAF 500
  [plate,*]: FRA 100  DET 100  LAN 0    WIN 50   STL 200  FRE 300  LAF 250
  ;

#### Example 3 — Multiperiod production with market areas

This is the `steelT3.mod` pattern from Fourer, Gay, Kernighan (Figure 6-3). Each product
is sold in a different set of geographic areas. The indexed family `AREA {PROD}` captures
this irregular structure. Parameters and variables that depend on both the product and
the area are indexed using `{p in PROD, AREA[p], 1..T}` — the second phrase
`AREA[p]` depends on the first dummy index `p`, which is standard AMPL idiom for
hierarchical indexing.

In [ ]:
# steelT3.mod  (Fourer, Gay, Kernighan, Figure 6-3)

set PROD;                               # products
set AREA {PROD};                        # market areas for each product
param T > 0;                            # number of weeks

param rate     {PROD}                    > 0;   # tons per hour produced
param inv0     {PROD}                   >= 0;   # initial inventory
param avail    {1..T}                   >= 0;   # hours available in each week
param prodcost {PROD}                   >= 0;   # cost per ton produced
param invcost  {PROD}                   >= 0;   # carrying cost per ton of inventory
param revenue  {p in PROD, AREA[p], 1..T} >= 0; # revenue per ton sold
param market   {p in PROD, AREA[p], 1..T} >= 0; # maximum tons sellable

var Make {PROD, 1..T}               >= 0;
var Inv  {PROD, 0..T}               >= 0;
var Sell {p in PROD, a in AREA[p], t in 1..T}
    >= 0, <= market[p,a,t];

maximize Total_Profit:
    sum {p in PROD, t in 1..T} (
        sum {a in AREA[p]} revenue[p,a,t] * Sell[p,a,t]
      - prodcost[p] * Make[p,t]
      - invcost[p]  * Inv[p,t]
    );

subject to Time {t in 1..T}:
    sum {p in PROD} (1/rate[p]) * Make[p,t] <= avail[t];

subject to Balance {p in PROD, t in 1..T}:
    Make[p,t] + Inv[p,t-1]
    = sum {a in AREA[p]} Sell[p,a,t] + Inv[p,t];

subject to Init_Inv {p in PROD}:
    Inv[p,0] = inv0[p];

In [ ]:
# --- steelT3.dat ---
data;

param T := 4 ;

set PROD := bands coils ;

# Indexed family: one area-set per product (ragged structure)
set AREA[bands] := east north ;
set AREA[coils] := east west export ;

param avail    :=  1 40  2 40  3 32  4 40 ;
param rate     :=  bands 200  coils 140 ;
param inv0     :=  bands  10  coils   0 ;
param prodcost :=  bands  10  coils  11 ;
param invcost  :=  bands 2.5  coils   3 ;

param revenue :=
  [bands,*,*]:   1     2     3     4 :=
      east     25.0  26.0  27.0  27.0
      north    26.5  27.5  28.0  28.5

  [coils,*,*]:   1     2     3     4 :=
      east      30    35    37    39
      west      29    32    33    35
      export    25    25    25    28 ;

param market :=
  [bands,*,*]:     1     2     3     4 :=
       east     2000  2000  1500  2000
       north    4000  4000  2500  4500

  [coils,*,*]:     1     2     3     4 :=
       east     1000   800  1000  1100
       west     2000  1200  2000  2300
       export   1000   500   500   800 ;

#### Iterated operators over indexed families

The `union` and `inter` reduction operators can iterate over all components of an indexed
family, producing a single flat set. This is the standard pattern when the model needs
to refer to "the set of all nodes reachable from any product's origin" or "the set of
origins common to all products".

| Expression | Result |
|---|---|
| `union {p in PROD} orig[p]` | All origins used by at least one product |
| `inter {p in PROD} orig[p]` | Origins used by every product (requires `PROD` non-empty) |
| `union {i in NODES} ARCS[i]` | Set of all nodes that appear as a destination of some arc |
| `card(ARCS[i])` | Number of arcs leaving node `i` (scalar integer) |

<div class="alert alert-block alert-warning"> <b>Pitfall: </b><br>
Iterated <code>inter</code> over an empty index set is undefined and causes a runtime error:
<em>"inter over empty set is not defined"</em>. Before using
<code>inter {i in S} F[i]</code>, ensure that <code>S</code> is non-empty — either by
a <code>check</code> statement or by the model's structural guarantees. Iterated
<code>union</code> over an empty index set returns the empty set without error.</div>

In [ ]:
# Iterated union and inter over an indexed family

set PROD;
set orig {PROD} within ORIG;
set dest {PROD} within DEST;

# All origins used by at least one product:
set ALL_ORIG = union {p in PROD} orig[p];

# All destinations used by at least one product:
set ALL_DEST = union {p in PROD} dest[p];

# Origins common to ALL products (requires PROD to be non-empty):
set COMMON_ORIG = inter {p in PROD} orig[p];

# Cardinality of each component:
param n_orig {p in PROD} = card(orig[p]);   # number of origins for product p
param n_dest {p in PROD} = card(dest[p]);   # number of destinations for product p

# Guard for iterated inter to avoid an empty-index error:
check card(PROD) > 0: "PROD must be non-empty for COMMON_ORIG to be defined";

#### Using indexed families in constraints: nested indexing

The most common use of indexed set families is in constraint indexing expressions where a
later phrase depends on a dummy variable defined by an earlier phrase. This produces
**nested indexing** — the inner set is different for each value of the outer index.

The comma-separated phrases in an indexing expression are evaluated left to right. A
dummy variable defined in one phrase is available in all subsequent phrases. This is what
makes `{i in NODES, j in ARCS[i]}` valid: `i` is defined first, then `ARCS[i]` is
evaluated for that specific `i`.

Three-level nesting — multi-hop paths — follows the same rule:

In [ ]:
# Two-level nesting: standard arc-indexed constraint
set NODES;
set ARCS {NODES} within NODES;
param cap {i in NODES, j in ARCS[i]} >= 0;
var   Flow {i in NODES, j in ARCS[i]} >= 0, <= cap[i,j];

# Three-level nesting: two-hop paths through the network
# {i in NODES, j in ARCS[i], k in ARCS[j]}
# reads: "for each node i, for each direct successor j of i,
#          for each direct successor k of j"
param two_hop_cap {i in NODES, j in ARCS[i], k in ARCS[j]} >= 0 default 0;
# two_hop_cap[i,j,k] is defined only for paths i -> j -> k in the graph.

# Constraint: at each intermediate node j, total flow arriving from i
# and continuing to k is limited.
subject to Path_Cap {i in NODES, j in ARCS[i]}:
    Flow[i,j] <= sum {k in ARCS[j]} cap[j,k];  # outbound capacity of j

# Membership test using the indexed family:
# Check whether arc (u,v) exists
param arc_exists {u in NODES, v in NODES} = if v in ARCS[u] then 1 else 0;

#### Multi-dimensional index: family indexed by pairs

The index of a set family need not be one-dimensional. Any valid indexing expression
— including a Cartesian product or a filtered set of pairs — may appear in the braces.
The resulting component is accessed with multiple subscripts.

In [ ]:
# Family indexed by (origin, destination) pairs:
# S[i,j] = set of products that can be shipped from origin i to destination j

set ORIG;
set DEST;
set PROD;

set S {ORIG, DEST} within PROD;        # S[i,j]: products on link (i,j)

param cost {i in ORIG, j in DEST, S[i,j]} >= 0;  # cost only for valid (i,j,p)
var   Ship {i in ORIG, j in DEST, S[i,j]} >= 0;

minimize Total_Cost:
    sum {i in ORIG, j in DEST, p in S[i,j]} cost[i,j,p] * Ship[i,j,p];

# Data assignment for a pair-indexed family:
# data;
# set S[GARY, DET] := bands coils ;
# set S[GARY, LAF] := bands ;
# set S[CLEV, DET] := coils plate ;
# (etc.)

#### `within` on indexed families

The `within` attribute on an indexed set family constrains **each component individually**.
The right-hand side of `within` may reference the indexing dummy variable(s), making
it possible to say "each `ARCS[i]` must be a subset of `NODES`" or, more specifically,
"each component must be a subset of a set that itself depends on the index".

In [ ]:
# within a fixed set:
set NODES;
set ARCS {NODES} within NODES;              # each ARCS[i] must be a subset of NODES

# within a set that depends on the index:
# (hypothetical: valid successors of i are only nodes with higher index)
set NODES2 ordered;
set FWD_ARCS {i in NODES2} within {j in NODES2: ord(j) > ord(i)};

# within for a compound component set:
# Each ROUTE[p] must be a subset of ORIG cross DEST
set ORIG2;
set DEST2;
set PROD2;
set ROUTE {PROD2} within ORIG2 cross DEST2;  # dimen 2 inferred from 'within' expression

<div class="alert alert-block alert-success"> <b>Best practice: </b><br>
Always include a <code>within</code> attribute when declaring an indexed set family whose
components must be subsets of a known universe. AMPL validates each data-supplied component
against the <code>within</code> bound at data-load time, producing an informative error
immediately rather than a silent wrong answer at solve time. Without <code>within</code>,
a mis-typed node name in a data file will be silently accepted as a new set member.</div>

<div class="alert alert-block alert-success"> <b>Best practice: </b><br>
Use <code>= sexpr</code> for any component that can be computed from other indexed
families rather than supplying it in data. The declaration
<code>set links {p in PROD} = orig[p] cross dest[p];</code> guarantees that
<code>links[p]</code> is always consistent with <code>orig[p]</code> and <code>dest[p]</code>
— no synchronisation across three data sections is required.</div>

<div class="alert alert-block alert-warning"> <b>Pitfall: </b><br>
An indexed set family declared with <code>=</code> cannot receive data from a <code>set</code>
statement in a <code>.dat</code> file or a <code>let</code> command. Attempting to assign
data to <code>links</code> (declared with <code>= orig[p] cross dest[p]</code>) produces an
AMPL error. If the set must be overridable, use <code>default</code> instead of
<code>=</code>.</div>

<div class="alert alert-block alert-warning"> <b>Pitfall: </b><br>
When accessing a component of an indexed family, the subscript type must match the
indexing set. If <code>ARCS</code> is indexed over <code>NODES</code> (a set of strings),
writing <code>ARCS[1]</code> (a number) instead of <code>ARCS['a']</code> causes an
<em>invalid subscript</em> error. AMPL checks subscript membership at model-generation time.
</div>

<div class="alert alert-block alert-warning"> <b>Pitfall: </b><br>
Do not confuse the indexed collection <code>set ARCS {NODES}</code> with the set of pairs
<code>set ALL_ARCS within NODES cross NODES</code>. The first declares one set per node;
the second declares a single flat 2-D set. They can represent the same graph, but
<code>ARCS cross NODES</code> is a syntax error because <code>ARCS</code> is an indexed
collection, not a set expression. To flatten an indexed family to a set of pairs, use
<code>setof {i in NODES, j in ARCS[i]} (i,j)</code>.</div>

#### amplpy equivalents

In amplpy, an indexed set family is assigned by iterating over the index set and setting
each component individually, or by passing a Python dictionary keyed by the index values.

| Operation | amplpy syntax | Notes |
|---|---|---|
| Assign one component | `ampl.set["ARCS"]["a"] = ["b", "c"]` | Key must be a valid member of `NODES` |
| Assign all components from dict | `ampl.set["AREA"] = {"bands": ["east","north"], "coils": ["east","west","export"]}` | Dict keyed by index values |
| Assign all components via loop | `for p in PROD: ampl.set["orig"][p] = orig_data[p]` | Equivalent to dict form |
| Read one component | `ampl.set["ARCS"]["a"].members()` | Returns a Python list |
| Read all components | `{i: ampl.set["ARCS"][i].members() for i in ampl.set["NODES"].members()}` | Reconstruct as Python dict |
| Cardinality of one component | `ampl.get_value("card(ARCS['a'])")` | Returns a float; cast to int |
| Iterated union in AMPL | `ampl.eval("display union {p in PROD} orig[p];")` | Evaluated inside AMPL |
| Iterated union result in Python | `ampl.get_set("ALL_ORIG").members()` | Requires `ALL_ORIG` declared in model |

In [ ]:
# amplpy: loading an indexed set family

from amplpy import AMPL

ampl = AMPL()

# Define the model inline
ampl.eval("""
    set PROD;
    set ORIG;
    set DEST;
    set orig {PROD} within ORIG;
    set dest {PROD} within DEST;
    set links {p in PROD} = orig[p] cross dest[p];
    set ALL_ORIG = union {p in PROD} orig[p];
""")

# Supply the flat sets
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.set["ORIG"] = ["GARY", "CLEV", "PITT"]
ampl.set["DEST"] = ["FRA", "DET", "LAN", "WIN", "STL", "FRE", "LAF"]

# Pattern 1: assign the entire indexed family from a dict (recommended)
orig_data = {
    "bands": ["GARY", "CLEV", "PITT"],
    "coils": ["GARY", "CLEV"],
    "plate": ["CLEV", "PITT"],
}
ampl.set["orig"] = orig_data   # dict-of-lists: key = index value, value = members

# Pattern 2: assign component by component (useful when data is built incrementally)
dest_data = {
    "bands": ["FRA", "DET", "LAN", "WIN", "STL", "FRE", "LAF"],
    "coils": ["FRA", "DET", "LAN", "WIN", "STL", "FRE", "LAF"],
    "plate": ["FRA", "DET", "WIN", "STL", "FRE", "LAF"],
}
for p, destinations in dest_data.items():
    ampl.set["dest"][p] = destinations

# Read back one component:
bands_orig = ampl.set["orig"]["bands"].members()   # ['GARY', 'CLEV', 'PITT']

# Read back the derived flat set:
all_orig = ampl.set["ALL_ORIG"].members()           # all origins across all products

print("orig[bands]:", bands_orig)
print("ALL_ORIG:",    sorted(all_orig))

#### Edge cases

**Edge case 1: An empty component set is valid and generates zero constraint instances.**

Any component of an indexed family may be the empty set. Constraints and parameters
indexed over an empty component are silently skipped — AMPL generates zero instances.
This is not an error. The `default {}` attribute is the canonical way to declare that
missing data should default to the empty set rather than trigger a warning.

In [ ]:
# Edge case 1: empty component set

set PROD;
set AREA {PROD} default {};     # empty set when data omits a component

param T > 0;
param revenue {p in PROD, AREA[p], 1..T} >= 0;
var   Sell    {p in PROD, a in AREA[p], t in 1..T} >= 0;

# If AREA[p] = {} for some product p, no Sell[p,*,*] variables are created
# and no revenue[p,*,*] data is required for that product.

# Data: only supply AREA for products that have market areas
data;
set PROD := bands coils wire ;
param T := 4 ;
set AREA[bands] := east north ;
set AREA[coils] := east west ;
# AREA[wire] is omitted => defaults to {}; no error, no Sell[wire,*,*] variables

**Edge case 2: Symbolically defined component using a conditional expression.**

The `=` expression for a component may be any valid set expression, including
conditionals and set operators. This allows components to be computed from data in
sophisticated ways. Recursive definitions of indexed sets are also allowed, provided
AMPL can compute them in a sequence that only references previously computed values
(AMPL Reference Manual, A.6).

In [ ]:
# Edge case 2: symbolically defined components using conditionals and recursion

# Example A: conditional definition
set PROD;
param premium {PROD} binary;    # 1 if product is a premium product

set PREMIUM_MARKETS = {"luxury", "specialty"};
set STD_MARKETS     = {"bulk", "discount"};

# Each product's market set is determined by its premium flag:
set MKTG {p in PROD} =
    if premium[p] = 1 then PREMIUM_MARKETS
    else STD_MARKETS;
# MKTG[p] is never supplied in data; it is always derived.

# Example B: recursive indexed set (transitive closure, from AMPL Reference Manual A.6)
set nodes;
set arcs within nodes cross nodes;
param max_iter = card(nodes) - 1;

# step[s] = set of pairs (i,j) reachable in exactly s hops or fewer
set step {s in 1..max_iter} dimen 2 =
    if s = 1
    then arcs
    else step[s-1] union
         setof {k in nodes, (i,k) in step[s-1], (k,j) in step[s-1]} (i,j);

set reach = step[max_iter];     # pairs reachable by any number of hops

**Edge case 3: Flattening an indexed family to a set of tuples with `setof`.**

When a constraint or operator needs all `(i,j)` pairs across all components of an indexed
family simultaneously — rather than iterating per-index — use `setof` to produce a
flat 2-D set. The `cross` operator cannot be used on an indexed family directly.

In [ ]:
# Edge case 3: flattening with setof

set NODES;
set ARCS {NODES} within NODES;

# Flatten the indexed family to a single 2-D set of pairs:
set ALL_ARCS = setof {i in NODES, j in ARCS[i]} (i,j);

# This is now a standard 2-D set and can be used with any set operator:
set NODES2;
set ARCS2 {NODES2} within NODES2;
set ALL_ARCS2 = setof {i in NODES2, j in ARCS2[i]} (i,j);

# Union of two flattened arc sets (from two separate graphs on the same nodes):
set COMBINED_ARCS = ALL_ARCS union ALL_ARCS2;   # valid: both are 2-D sets

# WRONG — ARCS is an indexed family, not a set expression:
# set BAD = NODES cross ARCS;   # ERROR: ARCS is not a set expression

# Membership test across the whole family:
# (u,v) in ALL_ARCS  <==>  v in ARCS[u]
param arc_exists {u in NODES, v in NODES} =
    if (u,v) in ALL_ARCS then 1 else 0;

**Edge case 4: Indexed family with a 2-dimensional index vs. a 2-dimensional component.**

Two independent sources of multi-dimensionality can arise:
1. The **index** of the family is multi-dimensional (e.g., indexed by pairs).
2. The **component** sets are themselves multi-dimensional (e.g., each component is a
   set of pairs).

These are independent and may be combined. Care is needed to distinguish subscripting
into the index (selecting which component) from subscripting into the component (selecting
a member).

In [ ]:
# Edge case 4: 2-D index vs. 2-D component

set ORIG;
set DEST;
set PROD;

# Case A: 2-D index, 1-D components
# S[i,j] = set of products viable on link (i,j)
set S {ORIG, DEST} within PROD;           # indexed by (orig,dest), component is 1-D
# Access: S["GARY","DET"] is a 1-D set of product names

# Case B: 1-D index, 2-D components
# T[p] = set of (orig,dest) pairs viable for product p
set T {PROD} within ORIG cross DEST;      # indexed by prod, component is 2-D (dimen 2)
# Access: T["bands"] is a 2-D set of (ORIG,DEST) pairs

# Case C: both 2-D index and 2-D component
# U[i,j] = set of (prod, period) pairs valid on link (i,j)
set PERIODS;
set U {ORIG, DEST} dimen 2 within PROD cross PERIODS;
# Access: U["GARY","DET"] is a 2-D set of (PROD,PERIODS) pairs

# Using T[p] in a parameter declaration:
param cost_T {p in PROD, T[p]} >= 0;      # cost[p,i,j] for (i,j) in T[p]
# This is equivalent to the multic.mod 'links' pattern above.

#### Cross-references

- See also: [1.1.1. Set Declaration vs. Definition](<1_1_1 Set Declaration vs. Definition.ipynb>) — the `=`, `default`, and `:=` operators
- See also: [1.1.2. Indexing expressions](<1_1_2 Indexing expressions.ipynb>) — the comma-separated indexing expression syntax and dummy variable scoping rules
- See also: [1.1.4. Set operations](<1_1_4 set_operations.ipynb>) — `union`, `inter`, `setof`, and `cross` used to build and combine sets
- See also: [1.1.5. Unordered sets](<1_1_5 unordered_sets.ipynb>) — ordering attributes, `within`, `dimen`
- AMPL book, Chapter 6 (Fourer, Gay, Kernighan): *Compound Sets and Indexing*, Section 6.5: *Indexed Collections of Sets*
- AMPL Reference Manual, Section A.6: *Set declarations* — formal grammar for indexed set declarations